In [19]:
from qiskit import * 
import matplotlib.pyplot as plt
from qiskit_aer import Aer

n_qubits = 10 # Number of qubits

qc = QuantumCircuit(n_qubits, n_qubits) # Create a quantum circuit with n_qubits qubits and n_qubits classical bits

qc.h(range(n_qubits))

qc.measure(range(n_qubits), range(n_qubits))

simulator = Aer.get_backend('qasm_simulator')
qc_transpiled = transpile(qc, simulator)
results = simulator.run(qc_transpiled, backend=simulator, shots=1000).result()
counts = results.get_counts()

print("Results from the quantum circuit")
print(counts)

Results from the quantum circuit
{'1010010110': 3, '1111100111': 2, '1011101011': 1, '0111000001': 2, '1001101110': 1, '0000001010': 2, '1010110001': 1, '1011001111': 2, '1101010001': 4, '1110111101': 1, '1000110110': 2, '0110101100': 3, '1001011101': 2, '1111010110': 1, '0110001110': 1, '0101100111': 2, '1101011110': 1, '1011000010': 1, '1001010000': 1, '0110100111': 1, '1111011011': 1, '1110110000': 3, '1000111011': 1, '0111011000': 1, '0010110110': 3, '0101001110': 2, '0010100001': 1, '0000001000': 1, '0001011110': 1, '0010110000': 2, '0101111101': 1, '1000100000': 3, '1100101110': 1, '1111100000': 2, '1011010100': 1, '1100001010': 1, '1100110100': 2, '1111011110': 1, '1001010111': 2, '1101111000': 1, '1000111110': 1, '0011001110': 2, '0011100111': 2, '0110101000': 1, '0111110001': 1, '0001011111': 2, '0111111000': 1, '1010001000': 1, '1111010010': 1, '1100000110': 2, '1011011100': 6, '0010010100': 1, '1100111100': 2, '0010100111': 3, '0101001100': 1, '1011001001': 1, '1101001001': 

In [20]:
import numpy as np
from PIL import Image

side = int(np.ceil(np.sqrt(len(counts))))
image = np.zeros((side, side))

# Map bitstring to pixel brightness based on freq
for idx, (bitstring, freq) in enumerate(sorted(counts.items())):
    row = idx // side
    col = idx % side

    intensity = freq # Freq as brightness
    image[row, col] = intensity

# Normalize values to range [0, 255] for grayscale image
normalized_image = (image / np.max(image) * 255).astype(np.uint8)

# Create and show the image
img = Image.fromarray(normalized_image, mode='L')
img.show()

# Optional: save the image to file
# img.save("quantum_art_01.png")

### Bitstring-to-Pixel Mapping (Inefficient Approach)

In this initial version, we create a quantum circuit with `n` qubits and map the output bitstrings to pixels. Each bitstring represents a unique combination (e.g., `01011`), and its frequency determines pixel intensity.

However, this approach has a critical limitation:

#### Scalability Issue

- The number of unique bitstrings is `2^n`.
- To generate a full image (e.g., 256×256 = 65,536 pixels), we would need **16 qubits** to cover all pixel slots.
- Quantum simulators (and even real quantum hardware) become inefficient and noisy with many qubits.

#### Conclusion

This method is good for experimentation and small patterns but not viable for generating high-resolution or scalable quantum art.

We will solve this in the next section by **reusing qubits efficiently** to generate larger images without increasing the number of qubits.


In [13]:
from qiskit import *
from qiskit_aer import Aer
from PIL import Image
import numpy as np
import random

# Parameters
n_qubits = 2
image_size = (512, 512)
n_pixels = image_size[0] * image_size[1]
shots = n_pixels

# Create quantum circuit with Hadamard gates 
qc = QuantumCircuit(n_qubits, n_qubits)
qc.h(range(n_qubits))

# Apply rotation to each qubit
for i in range(n_qubits):
    qc.rx(i * 0.5, i)
    qc.ry(i * 0.5, i)

qc.measure(range(n_qubits), range(n_qubits))

# Execute the circuit
simulator = Aer.get_backend('qasm_simulator')
qc_transpiled = transpile(qc, simulator)
results = simulator.run(qc_transpiled, backend=simulator, shots=shots).result()
counts = results.get_counts()

print('COUNTS: ', counts)

# Flatten all bitstring results into a list
bitstrings = []
for bitstring, count in counts.items():
    bitstrings.extend([bitstring] * count)
random.shuffle(bitstrings)

# Convert bitstrings to grayscale intensities
def bitstring_to_intensity(bitstring):
    return int(bitstring, 2) / (2**n_qubits - 1) * 255

pixels = [bitstring_to_intensity(b) for b in bitstrings]
image_array = np.array(pixels, dtype=np.uint8).reshape(image_size)

# Create and show image
img = Image.fromarray(image_array, mode='L')
img.show()

COUNTS:  {'10': 96785, '11': 96990, '00': 34153, '01': 34216}


In [6]:
from qiskit import *
from qiskit_aer import Aer
from PIL import Image
import numpy as np
import random

# Parameters
n_qubits = 2
height, width = 512, 512
n_pixels = image_size[0] * image_size[1]
shots = n_pixels
max_val = 2**n_qubits - 1

def generate_bitstrings(base_angles, variation=0.1):
    # Create quantum circuit with Hadamard gates 
    qc = QuantumCircuit(n_qubits, n_qubits)
    
    for i in range(n_qubits):
        angle = base_angles[i] + random.uniform(-variation, variation)
        qc.ry(angle, i)

    qc.measure(range(n_qubits), range(n_qubits))

    # Execute the circuit
    simulator = Aer.get_backend('qasm_simulator')
    qc_transpiled = transpile(qc, simulator)
    results = simulator.run(qc_transpiled, backend=simulator, shots=shots).result()
    counts = results.get_counts()

    # Flatten all bitstring results into a list
    bitstrings = []
    for bitstring, count in counts.items():
        bitstrings.extend([bitstring] * count)
    random.shuffle(bitstrings)

    data = np.zeros((height, width), dtype=np.uint8)
    for i in range(height):
        for j in range(width):
            idx = i * width + j
            b = bitstrings[idx]
            value = int(b, 2) / max_val * 255
            data[i, j] = int(value)

    return data

# Base angles to use across all channels
base_angles = [random.uniform(0, 2 * np.pi) for _ in range(n_qubits)]

# Generate each channel
r_bits = generate_bitstrings(base_angles, variation=0.05)
g_bits = generate_bitstrings(base_angles, variation=0.10)
b_bits = generate_bitstrings(base_angles, variation=0.15)

# Stack to form RGB image
rgb_image = np.stack([r_bits, g_bits, b_bits], axis=-1)
img = Image.fromarray(rgb_image, mode='RGB')
img.show()

In [ ]:
import json
from datetime import datetime
import random
import numpy as np

# Nombre base (puede usarse para agrupar serie o colección)
artwork_title = "Quantum Dream"

# Creamos una semilla para reproducibilidad
seed = random.randint(10000, 99999)
random.seed(seed)
np.random.seed(seed)

# Generamos ángulos base para todos los canales
base_angles = [random.uniform(0, 2 * np.pi) for _ in range(n_qubits)]

# Generamos la imagen
r = generate_bitstrings(base_angles, variation=0.05)
g = generate_bitstrings(base_angles, variation=0.10)
b = generate_bitstrings(base_angles, variation=0.15)

rgb_image = np.stack([r, g, b], axis=-1)
img = Image.fromarray(rgb_image, mode='RGB')

# Guardamos la imagen
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"{artwork_title}_{timestamp}_{seed}"
img.save(f"{filename}.png")

# Guardamos metadatos en JSON
metadata = {
    "title": artwork_title,
    "seed": seed,
    "base_angles": base_angles,
    "variation_r": 0.05,
    "variation_g": 0.10,
    "variation_b": 0.15,
    "num_qubits": n_qubits,
    "size": [height, width],
    "date": timestamp
}

with open(f"{filename}.json", "w") as f:
    json.dump(metadata, f, indent=4)

print(f"🎨 Artwork saved as {filename}.png with metadata.")


🎨 Artwork saved as Quantum Dream_20250424_171214_57396.png with metadata.
